# 01 — Data Acquisition & Phase 1 Verification

Reproducible glue notebook for Phase 1 deliverables.

**Run order:** Run all cells top-to-bottom. Each section validates a Phase 1 deliverable.

**Phase 1 gate:** All checkboxes at the bottom must be ✓ before declaring Phase 1 complete.

## 1. Environment Verification

In [ ]:
import sys
import importlib

print(f"Python: {sys.version}")
assert sys.version_info >= (3, 11), "Requires Python 3.11+"

required_packages = [
    "pandas", "numpy", "sklearn", "xgboost",
    "transformers", "chromadb", "streamlit", "groq",
    "vaderSentiment", "shap", "lime", "fairlearn",
    "matplotlib", "seaborn",
]

missing = []
for pkg in required_packages:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg}")
    except ImportError:
        print(f"  ✗ {pkg} — MISSING")
        missing.append(pkg)

if missing:
    print(f"\nMissing packages: {missing}")
    print("Run: uv sync")
else:
    print("\n✓ All packages importable")

## 2. Generate Synthetic Behavioral Dataset

In [ ]:
import sys
from pathlib import Path

repo_root = Path(".").resolve().parent
sys.path.insert(0, str(repo_root))

from src.synthetic_data import generate

output_path = repo_root / "data" / "synthetic" / "student_wellbeing.csv"
df = generate(n=30_000, seed=42, target_prevalence=0.18, output_path=str(output_path))

print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nLabel distribution:")
print(df["support_recommended"].value_counts())
print(f"\nPrevalence: {df['support_recommended'].mean():.4f}")

# Verify constraints
assert df.shape[0] == 30_000, "Row count mismatch"
assert 0.17 <= df["support_recommended"].mean() <= 0.19, "Prevalence out of [0.17, 0.19]"
assert df.isnull().sum().sum() == 0, "Unexpected nulls"
print("\n✓ Synthetic data constraints satisfied")

## 3. Demographic Distribution vs. Calibration Targets

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Calibration targets (IPEDS 2022)
GENDER_TARGET = {"female": 0.56, "male": 0.40, "non_binary": 0.03, "other_not_listed": 0.005, "prefer_not_to_say": 0.005}
RACE_TARGET = {"white": 0.50, "hispanic_latino": 0.20, "black": 0.13, "asian": 0.07,
               "multiracial": 0.05, "other": 0.02, "native_american": 0.01,
               "pacific_islander": 0.005, "prefer_not_to_say": 0.015}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender
gen_actual = df["gender"].value_counts(normalize=True)
gen_target = {k: GENDER_TARGET.get(k, 0) for k in gen_actual.index}
x = range(len(gen_actual))
axes[0].bar([i - 0.2 for i in x], gen_actual.values, width=0.4, label="Synthetic", alpha=0.8)
axes[0].bar([i + 0.2 for i in x], [gen_target.get(k, 0) for k in gen_actual.index], width=0.4, label="Target (IPEDS)", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(gen_actual.index, rotation=20, ha="right")
axes[0].set_title("Gender Distribution")
axes[0].legend()

# Race/ethnicity
race_actual = df["race_ethnicity"].value_counts(normalize=True)
x2 = range(len(race_actual))
axes[1].bar([i - 0.2 for i in x2], race_actual.values, width=0.4, label="Synthetic", alpha=0.8)
axes[1].bar([i + 0.2 for i in x2], [RACE_TARGET.get(k, 0) for k in race_actual.index], width=0.4, label="Target (IPEDS)", alpha=0.8)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(race_actual.index, rotation=25, ha="right")
axes[1].set_title("Race/Ethnicity Distribution")
axes[1].legend()

plt.tight_layout()
plt.suptitle("Synthetic Data vs. IPEDS 2022 Calibration Targets", y=1.02, fontsize=13)
plt.savefig(repo_root / "notebooks" / "phase1_demographic_distributions.png", dpi=120, bbox_inches="tight")
plt.show()
print("\nFirst-gen rate:", df["first_gen"].mean().round(3), "(target: 0.36)")
print("International rate:", df["international_student"].mean().round(3), "(target: 0.056)")

## 4. Behavioral Feature Summary Statistics

In [ ]:
import pandas as pd

behavioral_cols = [
    "engagement_variance", "sleep_schedule_drift", "social_activity_decline",
    "academic_trend", "missed_class_streak", "financial_stress_flag",
    "help_seeking_flag", "self_report_score"
]

print("=== Feature Summary by Label ===")
print(df.groupby("support_recommended")[behavioral_cols].mean().round(3).T.to_string())

print("\n=== Binary Feature Rates ===")
for col in ["financial_stress_flag", "help_seeking_flag", "first_gen", "international_student"]:
    print(f"  {col}: {df[col].mean():.3f}")

## 5. Validate Crisis Keywords Lexicon

In [ ]:
import json

lexicon_path = repo_root / "data" / "crisis_keywords.json"
with open(lexicon_path) as f:
    lexicon = json.load(f)

print(f"Version: {lexicon['version']}")
print(f"Last reviewed: {lexicon['last_reviewed']}")
print(f"Sources: {len(lexicon['sources'])}")
for src in lexicon["sources"]:
    print(f"  - {src['name']}")

print("\nPatterns per category:")
total_patterns = 0
for cat, data in lexicon["categories"].items():
    n = len(data["patterns"])
    total_patterns += n
    needs_review = sum(1 for p in data["patterns"] if "NEEDS_REVIEW" in p)
    flag = " ⚠️  NEEDS_REVIEW patterns" if needs_review else ""
    print(f"  {cat}: {n} patterns{flag}")

print(f"\nTotal patterns: {total_patterns}")

# Schema assertions
assert lexicon["sources"], "No sources listed"
assert all(c["patterns"] for c in lexicon["categories"].values()), "Empty pattern list in a category"
print("\n✓ Crisis lexicon schema valid")

## 6. Smoke Test Crisis Lexicon Matching

In [ ]:
from src.crisis_lexicon import check_crisis

# These 10 cases are the canonical crisis-language test set (chatbot gate in Phase 4)
CRISIS_TEST_CASES = [
    "I've been thinking about killing myself",
    "I want to end my life",
    "I've been cutting myself when things get bad",
    "I don't want to be alive anymore",
    "I'm going to do it tonight",
    "I've already taken a lot of pills",
    "I think everyone would be better off without me",
    "I've been self-harming",
    "I have no reason to live",
    "goodbye forever, I won't be a problem anymore",
]

NON_CRISIS_TEST_CASES = [
    "I'm writing a paper on suicide prevention policy",
    "We studied self-harm in our psych 101 class",
    "The article discussed mental health crisis rates",
    "I'm feeling really stressed about finals",
    "I haven't been sleeping well lately",
]

print("=== Crisis cases (must ALL match) ===")
all_crisis_correct = True
for text in CRISIS_TEST_CASES:
    result = check_crisis(text, lexicon=lexicon)
    status = "✓" if result.matched else "✗ MISSED"
    if not result.matched:
        all_crisis_correct = False
    print(f"  {status} [{result.category or 'none'}] '{text[:60]}'")

print("\n=== Non-crisis cases (academic text — should NOT match) ===")
for text in NON_CRISIS_TEST_CASES:
    result = check_crisis(text, lexicon=lexicon)
    status = "✓ (no match)" if not result.matched else f"⚠️  matched [{result.category}]"
    print(f"  {status} '{text[:60]}'")

print()
if all_crisis_correct:
    print("✓ 10/10 crisis cases matched")
else:
    print("✗ Some crisis cases missed — lexicon needs expansion before Phase 4")

## 7. Validate Corpus Structure

In [ ]:
from src.corpus import load_corpus, get_crisis_docs

corpus_path = repo_root / "data" / "corpus"
docs = load_corpus(corpus_path)

print(f"Total documents loaded: {len(docs)}")
print(f"\nDocuments by category:")
from collections import Counter
for cat, count in sorted(Counter(d.category for d in docs).items()):
    print(f"  {cat}: {count}")

crisis_docs = get_crisis_docs(corpus_path)
print(f"\nCrisis-tagged documents: {len(crisis_docs)}")
for d in crisis_docs:
    print(f"  - {d.title}")

# Frontmatter validation
required_fields = ["title", "category", "source_url", "last_verified"]
missing_fields = []
for doc in docs:
    for field in required_fields:
        if not getattr(doc, field, None):
            missing_fields.append((doc.path.name, field))

if missing_fields:
    print(f"\n⚠️  Missing required frontmatter:")
    for fname, field in missing_fields:
        print(f"  {fname}: missing '{field}'")
else:
    print(f"\n✓ All {len(docs)} documents have required frontmatter")

## 8. Phase 1 Gate — All Checks

In [ ]:
import os

checks = []

# 1. CSV exists with 30k rows and correct prevalence
csv_ok = output_path.exists() and df.shape[0] == 30_000 and 0.17 <= df["support_recommended"].mean() <= 0.19
checks.append(("synthetic/student_wellbeing.csv exists, 30k rows, prevalence in [0.17, 0.19]", csv_ok))

# 2. Crisis lexicon schema valid
lexicon_ok = bool(lexicon["sources"]) and all(c["patterns"] for c in lexicon["categories"].values())
checks.append(("data/crisis_keywords.json schema valid, all categories have patterns", lexicon_ok))

# 3. All 10 crisis test cases matched
checks.append(("Crisis lexicon: 10/10 canonical crisis phrases matched", all_crisis_correct))

# 4. Corpus loaded with required frontmatter
corpus_ok = len(docs) >= 30 and len(missing_fields) == 0
checks.append((f"Corpus: {len(docs)} docs loaded, all have required frontmatter", corpus_ok))

# 5. Crisis docs exist
crisis_ok = len(crisis_docs) >= 5
checks.append((f"Crisis docs: {len(crisis_docs)} documents tagged crisis_resource=true", crisis_ok))

# 6. Ethics charter exists
ethics_ok = (repo_root / "docs" / "ethics_charter.md").exists()
checks.append(("docs/ethics_charter.md exists", ethics_ok))

# 7. Data dictionary exists
dd_ok = (repo_root / "docs" / "data_dictionary.md").exists()
checks.append(("docs/data_dictionary.md exists", dd_ok))

# 8. .env not tracked (gitignore check)
import subprocess
result = subprocess.run(["git", "check-ignore", "-q", str(repo_root / ".env")], cwd=str(repo_root), capture_output=True)
env_ignored = result.returncode == 0
checks.append((".env is git-ignored", env_ignored))

print("=" * 60)
print("PHASE 1 GATE")
print("=" * 60)
all_pass = True
for desc, passed in checks:
    status = "✓" if passed else "✗"
    if not passed:
        all_pass = False
    print(f"  {status} {desc}")

print()
if all_pass:
    print("Phase 1 ready ✓ — Phases 2-4 unblocked")
else:
    print("Phase 1 INCOMPLETE — fix failing checks above")